# Weather collection, ranking and map exploration

Team Error_105 course project. See the repository README for inputs, working directories and scope.


In [ ]:
from pathlib import Path
Path('../../outputs/weather').mkdir(parents=True, exist_ok=True)


In [ ]:
import concurrent.futures
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json
import requests
from datetime import datetime, timedelta
import concurrent.futures
import time

In [ ]:
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json

nest_asyncio.apply()

# Load JSON file
file_path = "../../data/biomes_data/Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten data
# .items function returns a sort of list (dict_items data type that is non viewable that gives a iterable but not printable list of key-value pairs as tuples, so at top level I get biome and associated dictionaries) 
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]

# Fetch data for temperature and rainfall
async def fetch_forecast_data(session, coord):
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": coord["latitude"],
        "longitude": coord["longitude"],
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "timezone": "UTC",
    }

    retries = 3
    for attempt in range(retries):
        try:
            async with session.get(BASE_URL, params=params, timeout=15) as response:
                if response.status == 429:
                    retry_after = int(response.headers.get("Retry-After", 5))
                    await asyncio.sleep(retry_after)
                    continue
                response.raise_for_status()
                data = await response.json()
                if "daily" in data:
                    return {
                        "region": coord["region"],
                        "max_temperature": data["daily"]["temperature_2m_max"][0],
                        "min_temperature": data["daily"]["temperature_2m_min"][0],
                        "max_rainfall": data["daily"]["precipitation_sum"][0],
                    }
        except Exception as e:
            print(f"Error for {coord['region']}: {e}")
    return None

# Process all coordinates
async def process_all_coordinates():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_forecast_data(session, coord) for coord in coordinates]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    return [res for res in results if res]

# Main function
async def main():
    results = await process_all_coordinates()
    df = pd.DataFrame(results)

    hottest = df.sort_values(by="max_temperature", ascending=False).head(100)
    coldest = df.sort_values(by="min_temperature").head(100)
    wettest = df.sort_values(by="max_rainfall", ascending=False).head(100)

    hottest.to_csv("../../outputs/weather/hottest_places.csv", index=False)
    coldest.to_csv("../../outputs/weather/coldest_places.csv", index=False)
    wettest.to_csv("../../outputs/weather/wettest_places.csv", index=False)

# Run the async function
await main()


In [ ]:
filepath = '../../outputs/weather/coldest_places.csv'
df_coldest_places=pd.read_csv(filepath)


# Drop the 'max_temperature' column, as places are only ranked based on lowest temperature reached! (for ranking purposes)
df_coldest_places = df_coldest_places.drop(columns=['max_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_coldest_places = pd.merge(df_coldest_places, coordinates_df, on="region", how="left") # Left merge to retain df_coldest_places columns while adding coordinates for locations in top 100 where there is a match
df_coldest_places = df_coldest_places.rename(columns={'min_temperature': 'temperature'}) 
df_coldest_places.to_csv('../../outputs/weather/coldest_places.csv', index=False) # replace original CSV with updated one with coordinates!
print(df_coldest_places)

In [ ]:
filepath = '../../outputs/weather/hottest_places.csv'
df_hottest_places=pd.read_csv(filepath)


# Drop the 'min_temperature' column, only interested in max_temperature as that is basis of comparison
df_hottest_places = df_hottest_places.drop(columns=['min_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_hottest_places = pd.merge(df_hottest_places, coordinates_df, on="region", how="left")
df_hottest_places = df_hottest_places.rename(columns={'max_temperature': 'temperature'})
print(df_hottest_places)
df_hottest_places.to_csv('../../outputs/weather/hottest_places.csv', index=False)

In [ ]:
filepath = '../../outputs/weather/wettest_places.csv'
df_wettest_places=pd.read_csv(filepath)


df_wettest_places = df_wettest_places.drop(columns=["max_temperature", "min_temperature"]) #only interested in keeping max_rainfall
df_wettest_places = pd.merge(df_wettest_places, coordinates_df, on="region", how="left")
df_wettest_places.to_csv("../../outputs/weather/wettest_places.csv", index=False)

In [ ]:
import folium
from folium.plugins import HeatMap
from branca.colormap import linear

In [ ]:
# Initialize a base map (centered at a midpoint between the hottest and coldest locations)
m = folium.Map(location=[df_hottest_places['latitude'].mean(), df_hottest_places['longitude'].mean()], zoom_start=2)

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud')
    ).add_to(m)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud')
    ).add_to(m)

# Save or display the map
m.save("../../outputs/weather/temperature_map.html")
m


In [ ]:
import folium
from branca.colormap import linear
import pandas as pd





# Initialise the base map centered between hottest and coldest places
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create color scales for hot and cold places
hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())

# Add markers for hottest places with gradient colors
for idx, row in df_hottest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=hot_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add markers for coldest places with gradient colors
for idx, row in df_coldest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=cold_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add legends to the map
hot_colormap.caption = 'Temperature Scale (Hottest Places)'
cold_colormap.caption = 'Temperature Scale (Coldest Places)'
hot_colormap.add_to(m)
cold_colormap.add_to(m)

# Save or display the map
m.save("../../outputs/weather/temperature_ranked_map.html")
m


In [ ]:
import sqlite3


# Load CSV files
coldest_places = pd.read_csv("../../outputs/weather/coldest_places.csv")
hottest_places = pd.read_csv("../../outputs/weather/hottest_places.csv")
wettest_places = pd.read_csv("../../outputs/weather/wettest_places.csv") 

# Sort coldest places and assign unique ranking
coldest_places = coldest_places.sort_values(by="temperature", ascending=True)
coldest_places["ranking"] = range(1, len(coldest_places) + 1) # range(1,101) is used for rankings to be generated from 1-100, quirok of range function:)

# Sort hottest places and assign unique ranking
hottest_places = hottest_places.sort_values(by="temperature", ascending=False)
hottest_places["ranking"] = range(1, len(hottest_places) + 1)

# Sort wettest places and assign unique ranking
wettest_places = wettest_places.sort_values(by="max_rainfall", ascending=False)
wettest_places["ranking"] = range(1, len(wettest_places) + 1)

# Connect to SQLite database
conn = sqlite3.connect("../../outputs/weather/places.db")
cursor = conn.cursor()

# Drop existing tables if they exist
cursor.executescript("""
DROP TABLE IF EXISTS hottest_places;
DROP TABLE IF EXISTS coldest_places;
DROP TABLE IF EXISTS wettest_places;

CREATE TABLE hottest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    temperature FLOAT,
    latitude FLOAT,
    longitude FLOAT
);

CREATE TABLE coldest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    temperature FLOAT,
    latitude FLOAT,
    longitude FLOAT
);

CREATE TABLE wettest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    max_rainfall FLOAT,
    latitude FLOAT,
    longitude FLOAT
);
""")

# Insert data into tables
coldest_places[["ranking", "region", "temperature", "latitude", "longitude"]].to_sql("coldest_places", conn, if_exists="append", index=False)
hottest_places[["ranking", "region", "temperature", "latitude", "longitude"]].to_sql("hottest_places", conn, if_exists="append", index=False)
wettest_places[["ranking", "region", "max_rainfall", "latitude", "longitude"]].to_sql("wettest_places", conn, if_exists="append", index=False)

# Commit and close connection
conn.commit()
conn.close()

